# 🤖 X-MultiVLA: 비트코인 자동 트레이딩 봇
> **VLA(Vision-Language-Action)** 기반 강화학습 트레이딩 에이전트
>
> PatchTST(차트) + FinBERT(뉴스) + Cross-Attention + PPO

---
## 실행 순서
1. 🔧 환경 설치
2. 📁 Google Drive 마운트
3. 📥 데이터 수집
4. 🔄 전처리
5. 🧠 Phase 1: 지도학습 사전 학습
6. 🏋️ Phase 2: 강화학습 (PPO)
7. 📊 백테스트 & 평가
8. 💬 모델 설명 (XAI)
# ── 한글 폰트 설정 (차트 한글 깨짐 방지) ──────────────────
import subprocess, matplotlib.pyplot as plt, matplotlib.font_manager as fm
subprocess.run("apt-get install -y fonts-nanum > /dev/null 2>&1", shell=True)
fm._load_fontmanager(try_read_cache=False)
_nanum = [f for f in fm.findSystemFonts() if "Nanum" in f and "Gothic" in f and "Bold" not in f]
if _nanum:
    plt.rcParams['font.family'] = fm.FontProperties(fname=_nanum[0]).get_name()
plt.rcParams['axes.unicode_minus'] = False


In [8]:
# ─── CELL 1: 환경 설치 ───────────────────────────────────
!pip install -q ccxt yfinance fredapi stable-baselines3 shimmy transformers shap

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.2/141.2 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.8/223.8 kB 20.0 MB/s eta 0:00:00


In [9]:
# ─── CELL 2: Google Drive 마운트 ──────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, os
# 프로젝트 루트를 Python 경로에 추가
PROJECT_ROOT = '/content/drive/MyDrive/X-MultiVLA'
os.makedirs(PROJECT_ROOT, exist_ok=True)
sys.path.insert(0, PROJECT_ROOT)

# GPU 확인
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'💻 Device: {device}')
if device == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
💻 Device: cpu


In [11]:
# ─── CELL 3: 데이터 수집 (저장된 파일 우선 로드) ──────────
import os, glob, pandas as pd

RAW_DIR = f'{PROJECT_ROOT}/data/raw'

# 저장된 CSV 파일이 있으면 로드, 없으면 수집
csv_files = sorted(glob.glob(f'{RAW_DIR}/market_*.csv'))
if csv_files:
    latest = csv_files[-1]
    print(f'[collector] 저장된 데이터 로드: {latest}')
    df_raw = pd.read_csv(latest, index_col=0, parse_dates=True)
    # 인덱스를 UTC timezone-aware로 변환
    if df_raw.index.tz is None:
        df_raw.index = df_raw.index.tz_localize('UTC')
    print(f'\n로드된 데이터: {df_raw.shape}')
    print(df_raw.tail(3))
else:
    from data.collector import MarketDataCollector
    collector = MarketDataCollector()
    df_raw = collector.collect_all(timeframe='1d', days=730)
    collector.save(df_raw)
    print(f'\n수집된 데이터: {df_raw.shape}')
    print(df_raw.tail(3))

[collector] 저장된 데이터 로드: /content/drive/MyDrive/X-MultiVLA/data/raw/market_20260411.csv

로드된 데이터: (730, 8)
                           BTC_open  BTC_high   BTC_low  BTC_close  \
2026-04-09 00:00:00+00:00  71069.93  73145.00  70466.00   71787.97   
2026-04-10 00:00:00+00:00  71787.98  73434.00  71426.15   72962.70   
2026-04-11 00:00:00+00:00  72962.71  73094.89  72513.09   72615.55   

                            BTC_volume        DXY         GOLD        NASDAQ  
2026-04-09 00:00:00+00:00  18158.42159  98.820000  4792.200195  22822.419922  
2026-04-10 00:00:00+00:00  17372.63288  98.650002  4761.899902  22902.890625  
2026-04-11 00:00:00+00:00   4542.63169  98.650002  4761.899902  22902.890625  


In [12]:
# ─── CELL 4: 뉴스 데이터 수집 + 감성 분석 ──────────────
from data.news_fetcher import NewsFetcher

fetcher  = NewsFetcher()
raw_news = fetcher.fetch_raw(pages=10)          # 약 200개 뉴스
news_df  = fetcher.analyze_sentiment(raw_news)

# 차트 데이터와 시간 정렬
df_with_news = fetcher.align_with_chart(df_raw, news_df)
print(f'뉴스 포함 데이터: {df_with_news.shape}')
df_with_news[['news_sentiment_score']].tail()

[news] FinBERT 로딩 중...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[news] FinBERT 로드 완료 (device=cpu)
[news] CRYPTOPANIC_API_KEY 없음 → 더미 데이터 반환
[news] FinBERT 감성 분석 중... (100개)
[news] 감성 분석 완료
[news] 차트-뉴스 정렬 완료: shape=(730, 12)
뉴스 포함 데이터: (730, 12)


,news_sentiment_score
ts,
2026-04-07 00:00:00+00:00,-0.022591
2026-04-08 00:00:00+00:00,-0.022591
2026-04-09 00:00:00+00:00,-0.022591
2026-04-10 00:00:00+00:00,-0.022591
2026-04-11 00:00:00+00:00,-0.022591


In [13]:
# ─── CELL 5: 전처리 ─────────────────────────────────────
import numpy as np
from data.preprocessor import Preprocessor

pp = Preprocessor()
train_ds, val_ds, test_ds = pp.fit_transform(df_with_news)

# Scaler 저장
SCALER_PATH = f'{PROJECT_ROOT}/checkpoints/scaler.pkl'
pp.save_scaler(SCALER_PATH)

print(f'피처 수: {len(pp.feature_cols)}')
print(f'피처 목록: {pp.feature_cols[:8]} ...')

# DataLoader 생성
from torch.utils.data import DataLoader
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=2)

[preprocessor] 피처 수: 20
[preprocessor] train=508, val=11, test=12
[preprocessor] Scaler 저장: /content/drive/MyDrive/X-MultiVLA/checkpoints/scaler.pkl
피처 수: 20
피처 목록: ['BTC_open', 'BTC_high', 'BTC_low', 'BTC_close', 'BTC_volume', 'BTC_close_ret', 'NASDAQ_ret', 'GOLD_ret'] ...


In [14]:
# ─── CELL 6: Phase 1 - 지도학습 사전 학습 ───────────────
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from models.vla_agent import VLAPretrainer
from config import train_cfg, data_cfg

NUM_FEATURES = len(pp.feature_cols)
WINDOW       = data_cfg.window_size

pretrain_model = VLAPretrainer(num_features=NUM_FEATURES, window=WINDOW).to(device)
optimizer      = AdamW(pretrain_model.parameters(), lr=train_cfg.pretrain_lr, weight_decay=1e-4)
scheduler      = CosineAnnealingLR(optimizer, T_max=train_cfg.pretrain_epochs)
criterion      = nn.MSELoss()

PRETRAIN_CKPT = f'{PROJECT_ROOT}/checkpoints/pretrain_best.pt'
os.makedirs(os.path.dirname(PRETRAIN_CKPT), exist_ok=True)

best_val_loss = float('inf')
train_losses, val_losses = [], []

for epoch in range(train_cfg.pretrain_epochs):
    # ── Train ──
    pretrain_model.train()
    t_loss = 0
    for x_seq, y_true in train_loader:
        x_seq  = x_seq.to(device)          # (B, window, F)
        y_true = y_true.to(device)         # (B,)

        # 뉴스 벡터: 피처 중 news 컬럼 추출 (없으면 zeros)
        news_idx = [pp.feature_cols.index(c) for c in ['news_pos','news_neg','news_neu','news_sentiment_score'] if c in pp.feature_cols]
        if len(news_idx) == 4:
            news_vec = x_seq[:, -1, news_idx]   # 마지막 타임스텝의 뉴스 벡터
        else:
            news_vec = torch.zeros(x_seq.size(0), 4, device=device)

        pred   = pretrain_model(x_seq, news_vec).squeeze()
        loss   = criterion(pred, y_true)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(pretrain_model.parameters(), 1.0)
        optimizer.step()
        t_loss += loss.item()

    # ── Validation ──
    pretrain_model.eval()
    v_loss = 0
    with torch.no_grad():
        for x_seq, y_true in val_loader:
            x_seq  = x_seq.to(device)
            y_true = y_true.to(device)
            if len(news_idx) == 4:
                news_vec = x_seq[:, -1, news_idx]
            else:
                news_vec = torch.zeros(x_seq.size(0), 4, device=device)
            pred  = pretrain_model(x_seq, news_vec).squeeze()
            v_loss += criterion(pred, y_true).item()

    t_loss /= len(train_loader)
    v_loss /= len(val_loader)
    train_losses.append(t_loss)
    val_losses.append(v_loss)
    scheduler.step()

    if v_loss < best_val_loss:
        best_val_loss = v_loss
        torch.save(pretrain_model.state_dict(), PRETRAIN_CKPT)

    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:3d}/{train_cfg.pretrain_epochs} | Train: {t_loss:.6f} | Val: {v_loss:.6f}')

print(f'\n✅ 사전 학습 완료. Best Val Loss: {best_val_loss:.6f}')

# 학습 곡선 시각화
import matplotlib.pyplot as plt
plt.figure(figsize=(10,4))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses,   label='Val Loss')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.title('Phase 1: 사전 학습 손실 곡선')
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/content/drive/MyDrive/X-MultiVLA/models/encoders.py:94: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: Depreca

Epoch   5/30 | Train: 0.000626 | Val: 0.000530
Epoch  10/30 | Train: 0.000570 | Val: 0.000546
Epoch  15/30 | Train: 0.000565 | Val: 0.000612
Epoch  20/30 | Train: 0.000566 | Val: 0.000601
Epoch  25/30 | Train: 0.000553 | Val: 0.000606
Epoch  30/30 | Train: 0.000559 | Val: 0.000611

✅ 사전 학습 완료. Best Val Loss: 0.000516


/usr/local/lib/python3.12/dist-packages/IPython/core/pylabtools.py:151: UserWarning: Glyph 49324 (\N{HANGUL SYLLABLE SA}) missing from font(s) DejaVu Sans.
  fig.canvas.print_figure(bytes_io, **kw)
/usr/local/lib/python3.12/dist-packages/IPython/core/pylabtools.py:151: UserWarning: Glyph 51204 (\N{HANGUL SYLLABLE JEON}) missing from font(s) DejaVu Sans.
  fig.canvas.print_figure(bytes_io, **kw)
/usr/local/lib/python3.12/dist-packages/IPython/core/pylabtools.py:151: UserWarning: Glyph 54617 (\N{HANGUL SYLLABLE HAG}) missing from font(s) DejaVu Sans.
  fig.canvas.print_figure(bytes_io, **kw)
/usr/local/lib/python3.12/dist-packages/IPython/core/pylabtools.py:151: UserWarning: Glyph 49845 (\N{HANGUL SYLLABLE SEUB}) missing from font(s) DejaVu Sans.
  fig.canvas.print_figure(bytes_io, **kw)
/usr/local/lib/python3.12/dist-packages/IPython/core/pylabtools.py:151: UserWarning: Glyph 49552 (\N{HANGUL SYLLABLE SON}) missing from font(s) DejaVu Sans.
  fig.canvas.print_figure(bytes_io, **kw)
/usr

<Figure size 1000x400 with 1 Axes>

In [15]:
# ─── CELL 7: Phase 2 - PPO 강화학습 ─────────────────────
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
from stable_baselines3.common.vec_env import DummyVecEnv
from utils.gym_env import CryptoTradingEnv
from models.vla_agent import VLAAgent

# 사전 학습 모델 로드
pretrain_model.load_state_dict(torch.load(PRETRAIN_CKPT, map_location=device))

# 학습 / 검증 데이터 배열 (numpy)
df_eng      = pp.engineer_features(df_with_news)
X_all       = pp.scaler.transform(df_eng[pp.feature_cols].values)
prices_all  = df_eng['BTC_close'].values

n_train     = int(len(X_all) * data_cfg.train_ratio)
X_train_np  = X_all[:n_train]
p_train_np  = prices_all[:n_train]
X_val_np    = X_all[n_train:]
p_val_np    = prices_all[n_train:]

# Gym 환경 생성
def make_train_env():
    return CryptoTradingEnv(X_train_np, p_train_np, window=WINDOW)

def make_eval_env():
    return CryptoTradingEnv(X_val_np, p_val_np, window=WINDOW)

train_env   = DummyVecEnv([make_train_env])
eval_env    = DummyVecEnv([make_eval_env])

RL_CKPT     = f'{PROJECT_ROOT}/checkpoints/ppo_vla'
callbacks   = [
    EvalCallback(eval_env, best_model_save_path=f'{PROJECT_ROOT}/checkpoints',
                 log_path=f'{PROJECT_ROOT}/logs', eval_freq=5000, verbose=0),
    CheckpointCallback(save_freq=10000, save_path=f'{PROJECT_ROOT}/checkpoints', verbose=0),
]

# VLA 에이전트 생성 & 학습
agent = VLAAgent(
    env=train_env,
    num_features=NUM_FEATURES,
    window=WINDOW,
    pretrained_model=pretrain_model,
    device=device,
)

agent.train(total_timesteps=200_000, callback=callbacks)   # Colab 테스트용 20만 스텝
agent.save(RL_CKPT)
print(f'✅ PPO 학습 완료 → {RL_CKPT}')

Using cpu device
[VLAAgent] PPO 학습 시작 (200,000 timesteps)
Logging to /content/drive/MyDrive/X-MultiVLA/logs/PPO_1


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/policies.py:486: UserWarning: As shared layers in the mlp_extractor are removed since SB3 v1.8.0, you should now pass directly a dictionary and not a list (net_arch=dict(pi=..., vf=...) instead of net_arch=[dict(pi=..., vf=...)])
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


-----------------------------
| time/              |      |
|    fps             | 330  |
|    iterations      | 1    |
|    time_elapsed    | 6    |
|    total_timesteps | 2048 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 126         |
|    iterations           | 2           |
|    time_elapsed         | 32          |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.011074144 |
|    clip_fraction        | 0.121       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.09       |
|    explained_variance   | -0.19       |
|    learning_rate        | 0.0003      |
|    loss                 | -0.0376     |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0137     |
|    value_loss           | 0.0288      |
-----------------------------------------


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


------------------------------------------
| eval/                   |              |
|    mean_ep_length       | 83           |
|    mean_reward          | -0.757       |
| time/                   |              |
|    total_timesteps      | 5000         |
| train/                  |              |
|    approx_kl            | 0.0150626125 |
|    clip_fraction        | 0.145        |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.09        |
|    explained_variance   | 0.608        |
|    learning_rate        | 0.0003       |
|    loss                 | -0.0661      |
|    n_updates            | 20           |
|    policy_gradient_loss | -0.0161      |
|    value_loss           | 0.0145       |
------------------------------------------
-----------------------------
| time/              |      |
|    fps             | 102  |
|    iterations      | 3    |
|    time_elapsed    | 59   |
|    total_timesteps | 6144 |
-----------------------------
----------------

In [16]:
# ─── CELL 8: 백테스트 & 성과 분석 ───────────────────────
from utils.evaluator import Backtester

# 테스트 데이터
n_val   = int(len(X_all) * data_cfg.val_ratio)
X_test  = X_all[n_train + n_val:]
p_test  = prices_all[n_train + n_val:]

test_env = CryptoTradingEnv(X_test, p_test, window=WINDOW)

# 에이전트 로드 (또는 위 셀의 agent 직접 사용)
from models.vla_agent import VLAAgent
agent = VLAAgent.load(RL_CKPT + '.zip', env=DummyVecEnv([lambda: CryptoTradingEnv(X_test, p_test, window=WINDOW)]))

bt = Backtester(agent, X_test, p_test, window=WINDOW)
result = bt.run()

bt.print_metrics(result)
bt.plot(result, save_path=f'{PROJECT_ROOT}/outputs/backtest_result.png')


  X-MultiVLA 백테스트 성과 요약
  total_return        : 8.60%
  sharpe_ratio        : 39.15
  max_drawdown        : 2.02%
  win_rate            : 58.33%
  cagr                : 140535582340036190713937920.00%
  total_trades        : 12
  final_balance       : $10,860
[evaluator] 차트 저장: /content/drive/MyDrive/X-MultiVLA/outputs/backtest_result.png


<Figure size 1400x1000 with 3 Axes>

<Figure size 1400x1000 with 3 Axes>

In [17]:
# ─── CELL 9: XAI - 모델 설명 생성 ───────────────────────
from models.xai_explainer import VLAExplainer

explainer = VLAExplainer(pretrain_model, pp.feature_cols)

# 테스트 셋의 첫 번째 샘플로 설명 생성
sample_chart = torch.tensor(X_test[WINDOW:WINDOW+1], dtype=torch.float32).unsqueeze(0).to(device)
# (1, window, F) 형태로
sample_chart = torch.tensor(X_test[:WINDOW], dtype=torch.float32).unsqueeze(0).to(device)

obs, _ = test_env.reset()
action, _ = agent.predict(obs)

news_score = float(X_test[WINDOW - 1, pp.feature_cols.index('news_sentiment_score')]) if 'news_sentiment_score' in pp.feature_cols else 0.0

explanation = explainer.explain(
    chart_seq  = sample_chart,
    news_score = news_score,
    action     = int(action),
)
print(explanation)

🤖 X-MultiVLA 결정: 📈 Long (매수)

📊 [V] 차트 분석:
  · RSI -1.5 → 과매도 구간 (반등 가능성)
  · MACD Histogram 음수 → 하락 모멘텀
  · 직전 BTC 수익률: +46.87%

📰 [L] 뉴스·거시 분석:
  · 나스닥 하락(-68.12%) → 위험 회피 심리 ↑
  · 뉴스 감성: 중립(+0.00)

🎯 [A] 결론: Long (매수)


In [18]:
# ─── CELL 10: 피처 중요도 분석 ──────────────────────────
# 작은 배치로 permutation importance 계산
N = min(100, len(X_test) - WINDOW)
chart_batch = torch.tensor(
    np.array([X_test[i:i+WINDOW] for i in range(N)]), dtype=torch.float32
).to(device)
news_batch = torch.zeros(N, 4, device=device)

df_imp = explainer.feature_importance(chart_batch, news_batch, n_repeat=3, top_k=12)
print(df_imp.head(12).to_string(index=False))

/content/drive/MyDrive/X-MultiVLA/models/xai_explainer.py:198: UserWarning: Glyph 50696 (\N{HANGUL SYLLABLE YE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/content/drive/MyDrive/X-MultiVLA/models/xai_explainer.py:198: UserWarning: Glyph 52769 (\N{HANGUL SYLLABLE CEUG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/content/drive/MyDrive/X-MultiVLA/models/xai_explainer.py:198: UserWarning: Glyph 48320 (\N{HANGUL SYLLABLE BYEON}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/content/drive/MyDrive/X-MultiVLA/models/xai_explainer.py:198: UserWarning: Glyph 54868 (\N{HANGUL SYLLABLE HWA}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/content/drive/MyDrive/X-MultiVLA/models/xai_explainer.py:198: UserWarning: Glyph 47049 (\N{HANGUL SYLLABLE RYANG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/content/drive/MyDrive/X-MultiVLA/models/xai_explainer.py:198: UserWarning: Glyph 54588 (\N{HANGUL SYLLABLE PI}) missing from font(s) DejaVu Sans.
  pl

<Figure size 800x500 with 1 Axes>

      feature  importance
     GOLD_ret    0.060400
      BB_pctB    0.035002
      DXY_ret    0.020009
   volume_ret    0.018804
BTC_close_ret    0.018330
    MACD_hist    0.017135
   NASDAQ_ret    0.015214
   BTC_volume    0.014359
       RSI_14    0.012743
  MACD_signal    0.012265
        BB_bw    0.008876
         MACD    0.007406


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
